In [ ]:
# Connect to source data
!gdown --id 1ehw84L5fy6r3kZYzluq09Z2FJQvtzgVZ
!unzip -q super-ai-engineer-ss-6-heart-disease-prediction.zip

In [ ]:
# === CONFIGURATION ===
DATA_DIR = "/content"

In [ ]:
import pandas as pd
import numpy as np

# 1. โหลดข้อมูล
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. จัดการคอลัมน์ Target (ฉลากคำตอบ) ใน Train set
# ลบแถวที่ไม่มีคำตอบ (NaN) ออกจากข้อมูล Train
train_df = train_df.dropna(subset=['History of HeartDisease or Attack']).copy()

# แปลง Target จาก Yes/No ให้เป็น 1/0
train_df['History of HeartDisease or Attack'] = train_df['History of HeartDisease or Attack'].map({'Yes': 1, 'No': 0})

# แยก Features (X) และ Target (y)
X_train = train_df.drop(columns=['ID', 'History of HeartDisease or Attack'])
y_train = train_df['History of HeartDisease or Attack']
X_test = test_df.drop(columns=['ID'])

# 3. รวม X_train และ X_test ชั่วคราวเพื่อให้ Encoding ออกมาตรงกันทั้งสองชุด
combined_df = pd.concat([X_train, X_test], axis=0)

# 4. วนลูปจัดการทีละคอลัมน์
for col in combined_df.columns:
    if combined_df[col].dtype == 'object':

        # กรองดูว่าคอลัมน์นี้เป็นตัวแปรแบบ Yes/No หรือไม่
        unique_vals = combined_df[col].dropna().unique()
        if set(unique_vals).issubset({'Yes', 'No'}):
            combined_df[col] = combined_df[col].map({'Yes': 1, 'No': 0})

        # จัดการคอลัมน์เพศ
        elif col == 'Sex':
            combined_df[col] = combined_df[col].map({'Male': 1, 'Female': 0})

        # สำหรับตัวแปรอื่นๆ (Education, Income, General Health)
        # ให้แปลงเป็น Category เพื่อให้ LightGBM จัดการแบบ Native Categorical Feature
        else:
            combined_df[col] = combined_df[col].astype('category')

# 5. แยกข้อมูลกลับเป็น Train และ Test ตามสัดส่วนเดิม
X_train_clean = combined_df.iloc[:len(X_train)].copy()
X_test_clean = combined_df.iloc[len(X_train):].copy()

# 6. บังคับให้คอลัมน์ตัวเลขเป็นชนิด float เพื่อป้องกัน error ตอนเข้าโมเดล
for col in X_train_clean.columns:
    if X_train_clean[col].dtype != 'category':
        X_train_clean[col] = X_train_clean[col].astype(float)
        X_test_clean[col] = X_test_clean[col].astype(float)

print("Data Cleaning & Encoding Completed!")
print("Train Shape:", X_train_clean.shape)
print("Test Shape:", X_test_clean.shape)

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import fbeta_score

# กำหนด StratifiedKFold 5 Folds เพื่อรักษาสัดส่วนของคลาส
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_f2_scores = []
best_thresholds = []
# สร้าง array เก็บค่าความน่าจะเป็นสะสมสำหรับ Test set
test_preds_proba = np.zeros(len(X_test_clean))

# สร้างโมเดล LightGBM
clf = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    class_weight='balanced', # สำคัญมาก ช่วยเรื่อง Class Imbalance เบื้องต้น
    random_state=42,
    n_jobs=-1
)

print("Starting Training & Threshold Tuning...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_clean, y_train)):
    print(f"\n--- Fold {fold + 1} ---")

    # แบ่ง Data
    X_tr, y_tr = X_train_clean.iloc[train_idx], y_train.iloc[train_idx]
    X_va, y_va = X_train_clean.iloc[val_idx], y_train.iloc[val_idx]

    # ฝึกสอนโมเดล
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    # ทำนายความน่าจะเป็นของ Validation Set (ดึงเฉพาะคอลัมน์คลาส 1)
    val_preds_proba = clf.predict_proba(X_va)[:, 1]

    # วนลูปค้นหา Threshold ที่ดีที่สุดสำหรับ Fold นี้
    best_f2 = 0
    best_th = 0.5

    # ลอง Threshold ตั้งแต่ 0.10 ถึง 0.89 ขยับทีละ 0.01
    for th in np.arange(0.1, 0.9, 0.01):
        # ถ้าความน่าจะเป็นมากกว่า th ให้เป็น 1 (ป่วย) น้อยกว่าให้เป็น 0 (ไม่ป่วย)
        val_preds_binary = (val_preds_proba >= th).astype(int)

        # คำนวณ F2-Score (ตั้งค่า beta=2)
        current_f2 = fbeta_score(y_va, val_preds_binary, beta=2)

        if current_f2 > best_f2:
            best_f2 = current_f2
            best_th = th

    print(f"Best Threshold for Fold {fold + 1}: {best_th:.2f} | F2-Score: {best_f2:.4f}")

    oof_f2_scores.append(best_f2)
    best_thresholds.append(best_th)

    # ทำนาย Test set และบวกสะสมไว้ (หารด้วย 5 เพื่อหาค่าเฉลี่ยในตอนจบ)
    test_preds_proba += clf.predict_proba(X_test_clean)[:, 1] / skf.n_splits

print(f"\n==========================================")
print(f"Average F2-Score across 5 folds: {np.mean(oof_f2_scores):.4f}")
avg_threshold = np.mean(best_thresholds)
print(f"Average Best Threshold: {avg_threshold:.2f}")

# ---------------------------------------------------------
# การสร้างไฟล์ Submission
# ---------------------------------------------------------

# ใช้ Average Threshold ที่หามาได้ ตัดสินว่าใครป่วยหรือไม่ป่วย
final_test_preds = (test_preds_proba >= avg_threshold).astype(int)

# อ่านไฟล์ Sample Submission เป็นโครงร่าง
submission_df = pd.read_csv('sample_submission.csv')

# แปลง 1 เป็น 'Yes' และ 0 เป็น 'No' กลับคืน
submission_df['History of HeartDisease or Attack'] = np.where(final_test_preds == 1, 'Yes', 'No')

# บันทึกไฟล์
submission_df.to_csv("heart_disease_submission.csv", index=False)
print("Submission file 'heart_disease_submission.csv' created successfully!")